# ProofRush — Google Colab Cloud Compute Harness

ローカル Mac（Codex）の負荷を避け、**Google AI Pro の Colab Compute Units（月 200 Units）** で以下を実行するためのクラウド実行ノートブックです：
- **Tier A / B**: Lean 4 + Mathlib 並列検証、Python / Z3 / PySAT 大規模反例探索（推奨: **CPU ハイメモリ**）
- **Tier C**: 海洋ゴミ・環境数理モデルの実データ数値実験・統計検定（推奨: **GPU L4/T4**）

> **ポイント**: 初回のみ Mathlib キャッシュ（約6GB）を取得して Google Drive に圧縮保存します。2回目以降は Drive から高速展開（約15〜30秒）して NVMe 上で動作します。


## 1. Google Drive マウント & クラウド計算資源の確認


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!echo "=== CPU / RAM ==="
!nproc && free -h
!echo "=== GPU (if attached) ==="
!nvidia-smi 2>/dev/null || echo "CPU-only runtime (recommended for Lean 4 / SAT to save Compute Units)"
!echo "=== Disk ==="
!df -h /content


## 2. ProofRush リポジトリ取得 & Drive 永続領域の準備


In [ ]:
import os

REPO_URL = "https://github.com/yoshi0703/ProofRush.git"
WORK_DIR = "/content/ProofRush"
DRIVE_ROOT = "/content/drive/MyDrive/ProofRush_Cloud"

os.makedirs(f"{DRIVE_ROOT}/cache", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/exports", exist_ok=True)

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !git -C {WORK_DIR} pull --ff-only || true

%cd {WORK_DIR}
!ls -la


## 3. Lean 4 (`elan`) + Mathlib プリコンパイルキャッシュ高速セットアップ

Drive にキャッシュアーカイブがあれば展開し、初回は `lake exe cache get` で取得して Drive へ保存します（ソースからのフルビルドを回避）。


In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.elan/bin:$PATH"

DRIVE_CACHE="/content/drive/MyDrive/ProofRush_Cloud/cache"
WORK_DIR="/content/ProofRush"

# 1. elan インストール
if ! command -v elan >/dev/null 2>&1; then
  curl -sSf https://elan.lean-lang.org/elan-init.sh | sh -s -- -y --default-toolchain none
fi
export PATH="$HOME/.elan/bin:$PATH"
elan --version

# 2. zstd インストール（高速キャッシュ圧縮・展開用）
command -v zstd >/dev/null 2>&1 || (apt-get -qq update && apt-get -qq install -y zstd)

cd "${WORK_DIR}/Formal"

# 3. Drive キャッシュがあれば高速リストア、なければ lake exe cache get
CACHE_ARCHIVE="${DRIVE_CACHE}/mathlib_packages.tar.zst"
if [[ -f "${CACHE_ARCHIVE}" ]]; then
  echo "[Cache] Restoring Mathlib packages from Google Drive to fast local NVMe..."
  mkdir -p .lake
  tar -I zstd -xf "${CACHE_ARCHIVE}" -C .lake
  lake exe cache get || true
else
  echo "[Cache] First run: fetching precompiled Mathlib cache via lake exe cache get..."
  lake exe cache get
  echo "[Cache] Saving Mathlib packages archive to Google Drive for future sessions..."
  tar -I zstd -cf "${CACHE_ARCHIVE}" -C .lake packages
fi

echo "[Ready] Lean 4 + Mathlib cache ready on Colab!"


## 4. Python 科学計算・SAT/SMT ソルバ環境の導入（反例探索 & Tier C 環境科学用）


In [ ]:
!pip -q install --upgrade z3-solver python-sat sympy scipy pandas xarray netCDF4 statsmodels


## 5. スモークテスト（`worktree_ctl.sh` & `verify_gate.sh` のクラウド動作確認）


In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.elan/bin:$PATH"
cd /content/ProofRush

python3 scripts/state_ctl.py init
./scripts/worktree_ctl.sh create colab-smoke SmokeTest
./scripts/verify_gate.sh colab-smoke SmokeTest SmokeTest.main_conjecture
./scripts/worktree_ctl.sh remove colab-smoke
git branch -D attack/colab-smoke
rm -f state/locks/SmokeTest.sha256
echo "[OK] Cloud harness smoke test passed!"


## 6. クラウド実行結果を Google Drive へ退避するヘルパー

Colab のセッション切断前に `solved/` や `worker_report.json`、実験結果を Drive へ同期します。


In [ ]:
%%bash
set -euo pipefail
STAMP="$(date +%Y%m%d_%H%M%S)"
DEST="/content/drive/MyDrive/ProofRush_Cloud/exports/${STAMP}"
mkdir -p "${DEST}"

cp -r /content/ProofRush/solved "${DEST}/" 2>/dev/null || true
cp -r /content/ProofRush/EnvTrack "${DEST}/" 2>/dev/null || true
find /content/ProofRush/worktrees -maxdepth 2 -name "worker_report.json" -exec cp --parents {} "${DEST}/" \; 2>/dev/null || true

echo "[Synced] Exported artifacts to ${DEST}"
ls -la "${DEST}"
